<a href="https://colab.research.google.com/github/ali73/hands-on-mlp-exercise/blob/main/Season3/Ex3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data Fetch

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np

"""
Open a data set, with name, and load $sample_fraction$ of it.
returns X_train(sampled), Y_train(sampled), X_test, Y_test
dataset names: mnist: mnist_784
"""
def openDataset(name, as_frame = False, sample_fraction = 0.5):

  mnist = fetch_openml(name, as_frame=as_frame)
  X, Y = mnist.data, mnist.target
  X_train, X_test, Y_train, Y_test = train_test_split(
      X, Y, test_size=0.2, random_state=42, stratify=Y)

  # Define the fraction of the data to use (e.g., 50%)
  # Perform stratified sampling to ensure class balance
  sampled_indices = []
  # Convert Y_train to string for consistent type comparison, if needed
  Y_train_str = Y_train.astype(str)

  for class_label in np.unique(Y_train_str):
      class_indices = np.where(Y_train_str == class_label)[0]
      num_samples_in_class = int(len(class_indices) * sample_fraction)

      np.random.seed(42) # for reproducibility per class
      sampled_class_indices = np.random.choice(class_indices, num_samples_in_class, replace=False)
      sampled_indices.extend(sampled_class_indices)

  # Shuffle the final sampled indices to mix the classes
  np.random.shuffle(sampled_indices)

  X_train_sampled = X_train[sampled_indices]
  Y_train_sampled = Y_train[sampled_indices]
  return X_train_sampled, Y_train_sampled, X_test, Y_test


## Ex1

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

X_train, Y_train, X_test, Y_test = openDataset('mnist_784', sample_fraction=0.2)

# knc = KNeighborsClassifier(n_neighbors=10, weights = 'uniform', algorithm = 'kd_tree')
parameters = {
    'n_estimators': [50, 75, 100],
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 5, 10],
}
# cross_val_score(knc, X_train, Y_train, cv=3, scoring="accuracy", n_jobs=-1)

scoring = {
    "f1": "f1_macro",
    "precision": "precision_macro",
    "recall": "recall_macro"
}


gscv = GridSearchCV(estimator = model, param_grid = parameters, scoring = scoring,
                    refit = 'f1', cv = 3, n_jobs = -1)
gscv.fit(X_train, Y_train)
print(gscv.best_params_)
print(gscv.best_score_)


In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score

best_model = gscv.best_estimator_
y_pred = best_model.predict(X_test)

f1 = f1_score(Y_test, y_pred, average="macro")
acc = accuracy_score(Y_test, y_pred)
print(f"Test F1: {f1:.4f}")
print(f"Accuracy: {acc:.4f}")

## Ex 2

In [ ]:
from IPython.core.display import Image
X_train, Y_train, X_test, Y_test = openDataset('mnist_784', sample_fraction=0.2)

print(X_train.shape)
def shift_x(image, n):
    if n == 0: return image
    shape = list(image.shape)
    if n>=0:

      shifted_image = image[:, :-n] if n < len(image)  else np.array([], dtype=image.dtype)
      pad_shape = ( *shape[:1], n)
      padding = np.zeros(pad_shape, dtype=image.dtype)
      return np.concatenate([padding, shifted_image], axis=1)
    else :
      n = -n
      shifted_image = image[:, n:] if n < len(image)  else np.array([], dtype=image.dtype)
      pad_shape = (*shape[:1], n)
      padding = np.zeros(pad_shape, dtype=image.dtype)
      return np.concatenate([shifted_image, padding], axis=1)


example = np.array([[1,2,3,4], [5,6,7,8],[9,10,11,12]])
def shift_y(image, n):
  if n == 0:
    return image
  shape = list(image.shape)
  if n>=0:
    shifted_image = image[:-n, :] if n < len(image)  else np.array([], dtype=image.dtype)
    pad_shape = (n, *shape[1:])
    padding = np.zeros(pad_shape, dtype=image.dtype)
    return np.concatenate([padding, shifted_image], axis=0)
  else :
    n = -n
    shifted_image = image[n:, :] if n < len(image)  else np.array([], dtype=image.dtype)
    pad_shape = (n, *shape[1:])
    padding = np.zeros(pad_shape, dtype=image.dtype)
    return np.concatenate([shifted_image, padding], axis=0)


X_train = X_train.reshape((-1, 28, 28))
print(X_train.shape)

X_augmented = []
Y_augmented = []

for x, y in zip(X_train, Y_train):
  X_augmented.append(x)
  Y_augmented.append(y)

  X_augmented.append(shift_x(x, 1))
  Y_augmented.append(y)

  X_augmented.append(shift_x(x, -1))
  Y_augmented.append(y)

  X_augmented.append(shift_y(x, 1))
  Y_augmented.append(y)

  X_augmented.append(shift_y(x, -1))
  Y_augmented.append(y)\

X_augmented = np.array(X_augmented)
Y_augmented = np.array(Y_augmented)

print(X_augmented.shape)


In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score

best_model = gscv.best_estimator_
best_model.fit(X_augmented.reshape(-1, 784), Y_augmented)
y_pred = best_model.predict(X_test)

f1 = f1_score(Y_test, y_pred, average="macro")
acc = accuracy_score(Y_test, y_pred)
print(Y_test.shape)
print(f"Test F1: {f1:.4f}")
print(f"Accuracy: {acc:.4f}")

# Ex 3

**Import titanic files from kaggle**

In [20]:
import os
from google.colab import userdata

# Ensure the .kaggle directory exists
!mkdir -p ~/.kaggle

# Get the Kaggle API JSON from Colab secrets
kaggle_json = userdata.get('KAGGLE_CONFIG_JSON')

# Write the JSON content to kaggle.json file
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_json)

# Set permissions
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API key configured successfully!")

Kaggle API key configured successfully!


In [21]:
import kagglehub
# Download latest version
path = kagglehub.competition_download('titanic')

print("Path to competition files:", path)

Path to competition files: /root/.cache/kagglehub/competitions/titanic


In [22]:
import pandas as pd
train_df = pd.read_csv(os.path.join(path, 'train.csv'))
test_df = pd.read_csv(os.path.join(path, 'test.csv'))

print("\nFirst 5 rows of the test data:")
display(train_df.head())

print("\nInformation about the test data (columns, non-null counts, dtypes):")
train_df.info()


First 5 rows of the test data:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



Information about the test data (columns, non-null counts, dtypes):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [23]:
print(train_df.columns.to_list())

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [24]:
test_df.describe()

,PassengerId,Pclass,Age,SibSp,Parch,Fare
count,418.000000,418.000000,332.000000,418.000000,418.000000,417.000000
mean,1100.500000,2.265550,30.272590,0.447368,0.392344,35.627188
std,120.810458,0.841838,14.181209,0.896760,0.981429,55.907576
min,892.000000,1.000000,0.170000,0.000000,0.000000,0.000000
25%,996.250000,1.000000,21.000000,0.000000,0.000000,7.895800
50%,1100.500000,3.000000,27.000000,0.000000,0.000000,14.454200
75%,1204.750000,3.000000,39.000000,1.000000,0.000000,31.500000
max,1309.000000,3.000000,76.000000,8.000000,9.000000,512.329200


In [25]:
# Print some information about the features
sibsp_counts = train_df['SibSp'].value_counts().to_dict()
parch_counts = train_df['Parch'].value_counts().to_dict()
pclass_counts = train_df['Pclass'].value_counts().to_dict()

print("SibSp counts:", sibsp_counts)
print("Parch counts:", parch_counts)
print("Class counts:", pclass_counts)

SibSp counts: {0: 608, 1: 209, 2: 28, 4: 18, 3: 16, 8: 7, 5: 5}
Parch counts: {0: 678, 1: 118, 2: 80, 5: 5, 3: 5, 4: 4, 6: 1}
Class counts: {3: 491, 1: 216, 2: 184}


گروه‌های سنی ایجاد شده و به جای خود سن از آن‌ها استفاده شود.

In [26]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

age_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("binning", KBinsDiscretizer(
        n_bins=6,                 # Number of age groups
        encode="ordinal",         # Output: 0, 1, 2, ...
        strategy="kmeans"        # Equal-width bins
    ))
])
age = age_pipeline.fit_transform(train_df[['Age']])
kbd = age_pipeline.named_steps["binning"]
print(kbd.bin_edges_)
print(kbd.bin_edges_[0].shape)


[array([ 0.42      , 12.34322255, 24.17349743, 33.10139151, 43.36616385,
        56.11605136, 80.        ])                                      ]
(7,)


Convert categorical features to one-hot encoding

In [27]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline



# Create a pipeline for preprocessing categorical features
# It imputes missing values (for 'Embarked') and then applies one-hot encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Impute with most frequent for 'Embarked'
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])



In [28]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# Function to convert non-zero values to 1, and zero to 0
def to_binary(X):
    return (X != 0).astype(int)

# Create a pipeline for 'SibSp' and 'Parch' to convert them to 0-1 values
sibsp_parch_pipeline = Pipeline(steps=[
    ('to_binary', FunctionTransformer(to_binary, feature_names_out='one-to-one'))
])

Normalization of fares

In [29]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
import numpy as np

fare_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(lambda X: np.log1p(X), validate=False)),
    ("scaler", StandardScaler())
])

In [30]:
test_df.describe()

,PassengerId,Pclass,Age,SibSp,Parch,Fare
count,418.000000,418.000000,332.000000,418.000000,418.000000,417.000000
mean,1100.500000,2.265550,30.272590,0.447368,0.392344,35.627188
std,120.810458,0.841838,14.181209,0.896760,0.981429,55.907576
min,892.000000,1.000000,0.170000,0.000000,0.000000,0.000000
25%,996.250000,1.000000,21.000000,0.000000,0.000000,7.895800
50%,1100.500000,3.000000,27.000000,0.000000,0.000000,14.454200
75%,1204.750000,3.000000,39.000000,1.000000,0.000000,31.500000
max,1309.000000,3.000000,76.000000,8.000000,9.000000,512.329200


In [31]:
from sklearn.compose import ColumnTransformer

# Identify categorical columns for one-hot encoding
categorical_features = ['Sex', 'Embarked', 'Pclass']
# Identify numerical features for age binning
numerical_features = ['Age']
# Identify features for binary conversion
binary_features = ['SibSp', 'Parch']

# dropped_features = ["Name", "PassengerId"]
# We have no data about the ticket numbering system.
# dropped_features += ['Ticket']
# We have lots of missing data in cabin column
# dropped_features += ['Cabin']
fare_feature = ['Fare']
train_labels = train_df["Survived"]

# train_df.drop(columns=dropped_features, inplace=True)
# test_df.drop(columns=dropped_features, inplace=True)
# Create a ColumnTransformer to apply the categorical and numerical preprocessing
# 'remainder='passthrough'' keeps other columns as they are (e.g., PassengerId, Pclass, Fare)
preprocessor = ColumnTransformer(
    transformers=[
        ('age_bins', age_pipeline, numerical_features),
        ('cat', categorical_transformer, categorical_features),
        ('binary', sibsp_parch_pipeline, binary_features), # Add the new pipeline for SibSp and Parch
        ('fare', fare_pipeline, fare_feature)
    ],
    remainder='drop'
)


# Apply the preprocessing to the training data
train_df_encoded = preprocessor.fit_transform(train_df)

print("Shape of processed training data after one-hot encoding, age binning, and SibSp/Parch binary conversion:", train_df_encoded.shape)
# Display the first few rows of the encoded data (note: it's a numpy array now)
print("First 5 rows of encoded data:\n", train_df_encoded[:5])

Shape of processed training data after one-hot encoding, age binning, and SibSp/Parch binary conversion: (891, 12)
First 5 rows of encoded data:
 [[ 1.          0.          1.          0.          0.          1.
   0.          0.          1.          1.          0.         -0.87974057]
 [ 3.          1.          0.          1.          0.          0.
   1.          0.          0.          1.          0.          1.36121993]
 [ 2.          1.          0.          0.          0.          1.
   0.          0.          1.          0.          0.         -0.79853997]
 [ 3.          1.          0.          0.          0.          1.
   1.          0.          0.          1.          0.          1.06203806]
 [ 3.          0.          1.          0.          0.          1.
   0.          0.          1.          0.          0.         -0.78417924]]


In [32]:
train_df.drop(columns=['Survived'], inplace=True)

In [35]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

svc = SVC()
full_pipeline = Pipeline([
      ("preprocessing", preprocessor),
      ("svc", svc)
])

cvs = cross_val_score(full_pipeline, train_df, train_labels , cv=5, scoring="f1", n_jobs=-1)
print(cvs)

[0.73333333 0.73015873 0.75590551 0.69565217 0.75806452]


In [34]:
from sklearn.metrics import accuracy_score
full_pipeline.fit(train_df, train_labels)
y_pred = full_pipeline.predict(test_df)
